# P36 — Perdidos en el medio: cómo usan los modelos de lenguaje los contextos largos

## 1. Título y paper

**Paper:** *Lost in the Middle: How Language Models Use Long Contexts*  
**Autoría:** Nelson F. Liu, Kevin Lin, John Hewitt, Ashwin Paranjape, Michele Bevilacqua, Fabio Petroni, Percy Liang  
**Año y venue:** 2023 · arXiv:2307.03172 · TACL  
**Nivel:** L3 · **Motor:** `lost_in_middle`  
**Ficha completa:** [`P36_lost_in_middle`](../../papers/foundational/P36_lost_in_middle/README.md)

**Hito:** Tener contexto largo no es usarlo: el rendimiento cae en forma de U cuando el dato relevante está en el medio.

- [arXiv:2307.03172](https://arxiv.org/abs/2307.03172)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: La industria competía por anunciar ventanas de contexto cada vez mayores, sin medir si los modelos aprovechaban de verdad todo ese espacio.
2. Ejecutar una implementación mínima de la propuesta: Medirlo: colocar el mismo documento relevante en distintas posiciones del contexto y observar cómo cambia la exactitud.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- P10
- P11
- P35


## 4. Intuición

Le das al modelo veinte documentos y el dato bueno está en el número once. Rinde peor que si estuviera en el primero o en el último. El mismo dato, el mismo modelo, la misma pregunta: solo cambia el sitio.


## 5. Concepto mínimo

```text
exactitud(posición del documento relevante) tiene forma de U:

    alta al principio  (primacía)
    baja en el medio   ← el hallazgo
    alta al final      (recencia)
```


## 6. Código explicado

El motor aísla el mecanismo del paper con datos de juguete y salida inspeccionable.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('lost_in_middle', seed=7)['result']
show(r)

## 7. Predicción antes de ejecutar

1. ¿En qué posición esperas la mejor exactitud? ¿Y la peor?
2. ¿Qué implica esto para un sistema RAG que ordena los pasajes por score?
3. Si un modelo anuncia 128 000 tokens de contexto, ¿qué habría que medir antes de creerlo?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for semilla in (1, 7, 42):
    r = run_paper_lab('lost_in_middle', seed=semilla)
    print(f'semilla {semilla:>2} · evidencia principal:')
    for e in r['evidence']:
        print('   +', e)
    break  # determinista: basta una para ver la estructura
for semilla in (1, 7, 42):
    r = run_paper_lab('lost_in_middle', seed=semilla)['result']
    print(f'semilla {semilla:>2} → claves: {list(r)[:4]}')

## 9. Salida interpretable

La caída entre la mejor y la peor posición es grande, y no hay nada distinto en el contenido. **Contexto disponible no es contexto utilizable**, y esa distinción no aparece en ninguna ficha técnica de modelo.


## 10. Comentario pedagógico

El impacto práctico es directo en RAG: si recuperas diez pasajes y colocas el mejor en medio, estás saboteando tu propio sistema. Conviene poner lo más relevante al principio **o** al final, y medirlo en vez de suponerlo.


## 11. Error o anti-patrón deliberado

Anti-patrón: elegir modelo por el tamaño de su ventana de contexto.


In [ ]:
for ventana in (8_000, 32_000, 128_000, 1_000_000):
    print(f'{ventana:>9,} tokens anunciados → ¿cuantos USA bien? el numero no lo dice')

## 12. Corrección

La comprobación correcta es una prueba de aguja en el pajar por posición:


In [ ]:
protocolo = {'1': 'insertar un hecho unico en la posicion p del contexto',
             '2': 'preguntar por ese hecho',
             '3': 'repetir para p en todo el rango y varias longitudes',
             '4': 'reportar la CURVA, no un solo numero'}
show(protocolo)

## 13. Desafío guiado

Calcula la caída relativa entre la mejor y la peor posición y decide si es tolerable para un sistema de consulta legal.


In [ ]:
r = run_paper_lab('lost_in_middle', seed=3)['result']
show(r)

## 14. Desafío autónomo

Ejecuta una prueba de aguja en el pajar sobre un modelo abierto, con al menos cinco longitudes y diez posiciones. Dibuja la curva y localiza dónde empieza a degradarse.


## 15. Evidencia de aprendizaje

Guarda la curva en U, la caída entre extremos y tu protocolo de comprobación por posición.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P36_lost_in_middle/README.md) · evaluación formal: [`assessments/papers/P36_lost_in_middle.md`](../../assessments/papers/P36_lost_in_middle.md)


## 16. Cierre

Si la ventana no basta ni usándola bien, hay que dejar de tratarla como memoria y empezar a gestionarla como tal.


## 17. Conexión con el siguiente hito

- P37
- evaluación de contexto largo

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
